# Model Experiments: CatBoost and XGBoost

This notebook compares two regression approaches on the frozen October modeling split:

- CatBoostRegressor
- XGBRegressor

Both models use all 130 predictors for the initial comparison. Model-specific categorical handling is allowed, but the claim-level partitions and evaluation metric remain fixed.

The primary evaluation metric is MAE in original `loss` units. Cross-validation uses the five frozen training folds, and validation is used only as a limited comparison set.

After the full-feature comparison, feature importance is examined using gain-based importance, permutation importance, and SHAP. These measures will be aggregated into a feature ranking for later feature-selection experiments.

In [31]:
from pathlib import Path
import time

import numpy as np
import pandas as pd

from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error

from catboost import CatBoostRegressor
from xgboost import XGBRegressor

import hashlib

import shap

In [7]:
SEED = 42

CAT_COLS = [f"cat{i}" for i in range(1, 117)]
CONT_COLS = [f"cont{i}" for i in range(1, 15)]
FEATURE_COLS = CAT_COLS + CONT_COLS

SOURCE_SHA256 = "74037cb248a1064e4d578692a4f4e5d8492ed1b2033daf643496e1b68b14ae03"
MANIFEST_SHA256 = "de10b0e17ad56174a7b126352a0a41b3b3d477bbddd65f37b87a6235f582a64b"

TRAIN_COUNT = 131_822
VALIDATION_COUNT = 28_248

MEAN_BASELINE_MAE = 1962.918654312225
MEDIAN_BASELINE_MAE = 1805.0863526621356

CURRENT_BEST_MAE = 1151.17
CURRENT_BEST_MODEL = "HistGradientBoosting"

repo_root = Path.cwd()

while not (repo_root / "data" / "allstate_claims_data.csv").exists():
    if repo_root.parent == repo_root:
        raise FileNotFoundError("Could not locate repository root.")
    repo_root = repo_root.parent

data_path = repo_root / "data" / "allstate_claims_data.csv"
manifest_path = (
    repo_root
    / "notebooks"
    / "final-deliverables"
    / "October"
    / "Gate 1"
    / "partition_manifest.csv"
)

print("Repo root:", repo_root)

Repo root: c:\Users\stapl\BTT\Fall AI Studio\Allstate 1A\Allstate-1A-predicting-auto-claims-severity


In [8]:
def sha256_file(path):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)

    return digest.hexdigest()


source_hash = sha256_file(data_path)
manifest_hash = sha256_file(manifest_path)

assert source_hash == SOURCE_SHA256
assert manifest_hash == MANIFEST_SHA256

print("Frozen source and partition manifest verified.")

Frozen source and partition manifest verified.


In [9]:
data = pd.read_csv(data_path)
manifest = pd.read_csv(manifest_path)

df = data.merge(
    manifest,
    on="id",
    how="inner",
    validate="one_to_one",
)

train = df[df["partition"] == "train"].copy()
validation = df[df["partition"] == "validation"].copy()

assert len(train) == TRAIN_COUNT
assert len(validation) == VALIDATION_COUNT

X_train = train[FEATURE_COLS].copy()
y_train = train["loss"].copy()

X_valid = validation[FEATURE_COLS].copy()
y_valid = validation["loss"].copy()

train_ids = train["id"].copy()
validation_ids = validation["id"].copy()

cv_folds = train["cv_fold"].astype(int).copy()

print("Train:", X_train.shape)
print("Validation:", X_valid.shape)
print("CV folds:", sorted(cv_folds.unique()))

Train: (131822, 130)
Validation: (28248, 130)
CV folds: [0, 1, 2, 3, 4]


In [10]:
assert list(X_train.columns) == FEATURE_COLS
assert list(X_valid.columns) == FEATURE_COLS

assert X_train.isna().sum().sum() == 0
assert X_valid.isna().sum().sum() == 0

assert y_train.notna().all()
assert y_valid.notna().all()

assert set(cv_folds.unique()) == {0, 1, 2, 3, 4}

print("Input checks passed.")

Input checks passed.


## CatBoostRegressor

In [11]:
X_train_cat = X_train.copy()
X_valid_cat = X_valid.copy()

for col in CAT_COLS:
    X_train_cat[col] = X_train_cat[col].astype(str)
    X_valid_cat[col] = X_valid_cat[col].astype(str)

cat_feature_indices = [
    X_train_cat.columns.get_loc(col)
    for col in CAT_COLS
]

print("Categorical features:", len(cat_feature_indices))

Categorical features: 116


In [14]:
catboost_configs = [
    {
        "name": "catboost_1",
        "depth": 6,
        "learning_rate": 0.08,
        "iterations": 250,
    },
    {
        "name": "catboost_2",
        "depth": 8,
        "learning_rate": 0.08,
        "iterations": 250,
    },
]

In [15]:
catboost_cv_results = []

for config in catboost_configs:
    fold_maes = []
    fold_best_iterations = []
    start_time = time.time()

    print(f"\nRunning {config['name']}")

    for fold in range(5):
        fold_train_mask = cv_folds != fold
        fold_valid_mask = cv_folds == fold

        X_fold_train = X_train_cat.loc[fold_train_mask]
        y_fold_train = y_train.loc[fold_train_mask]

        X_fold_valid = X_train_cat.loc[fold_valid_mask]
        y_fold_valid = y_train.loc[fold_valid_mask]

        model = CatBoostRegressor(
            loss_function="MAE",
            eval_metric="MAE",
            depth=config["depth"],
            learning_rate=config["learning_rate"],
            iterations=config["iterations"],
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )

        model.fit(
            X_fold_train,
            y_fold_train,
            cat_features=cat_feature_indices,
            eval_set=(X_fold_valid, y_fold_valid),
            early_stopping_rounds=30,
            use_best_model=True,
        )

        predictions = model.predict(X_fold_valid)

        fold_mae = mean_absolute_error(
            y_fold_valid,
            predictions,
        )

        fold_maes.append(fold_mae)
        fold_best_iterations.append(model.get_best_iteration())

        print(
            f"  Fold {fold}: "
            f"MAE={fold_mae:.2f}, "
            f"best_iter={model.get_best_iteration()}"
        )

    runtime = time.time() - start_time

    catboost_cv_results.append(
        {
            **config,
            "cv_mae_mean": np.mean(fold_maes),
            "cv_mae_std": np.std(fold_maes),
            "mean_best_iteration": np.mean(fold_best_iterations),
            "runtime_seconds": runtime,
        }
    )

catboost_cv_df = pd.DataFrame(catboost_cv_results)

catboost_cv_df.sort_values("cv_mae_mean")


Running catboost_1
  Fold 0: MAE=1175.65, best_iter=249
  Fold 1: MAE=1168.18, best_iter=249
  Fold 2: MAE=1160.37, best_iter=249
  Fold 3: MAE=1174.25, best_iter=249
  Fold 4: MAE=1158.70, best_iter=249

Running catboost_2
  Fold 0: MAE=1169.11, best_iter=249
  Fold 1: MAE=1159.75, best_iter=249
  Fold 2: MAE=1154.37, best_iter=249
  Fold 3: MAE=1166.04, best_iter=249
  Fold 4: MAE=1150.71, best_iter=249


,name,depth,learning_rate,iterations,cv_mae_mean,cv_mae_std,mean_best_iteration,runtime_seconds
1,catboost_2,8,0.08,250,1159.997163,6.893022,249.0,1393.581232
0,catboost_1,6,0.08,250,1167.427961,6.937781,249.0,776.387510


In [16]:
best_catboost_config = (
    catboost_cv_df
    .sort_values("cv_mae_mean")
    .iloc[0]
)

best_catboost_config

name                    catboost_2
depth                            8
learning_rate                 0.08
iterations                     250
cv_mae_mean            1159.997163
cv_mae_std                6.893022
mean_best_iteration          249.0
runtime_seconds        1393.581232
Name: 1, dtype: object

In [17]:
catboost_start = time.time()

catboost_model = CatBoostRegressor(
    loss_function="MAE",
    eval_metric="MAE",
    depth=int(best_catboost_config["depth"]),
    learning_rate=float(best_catboost_config["learning_rate"]),
    iterations=int(best_catboost_config["iterations"]),
    random_seed=SEED,
    verbose=False,
    allow_writing_files=False,
)

catboost_model.fit(
    X_train_cat,
    y_train,
    cat_features=cat_feature_indices,
)

catboost_runtime = time.time() - catboost_start

catboost_valid_pred = catboost_model.predict(X_valid_cat)

catboost_valid_mae = mean_absolute_error(
    y_valid,
    catboost_valid_pred,
)

print(f"CatBoost validation MAE: {catboost_valid_mae:.2f}")
print(f"Runtime: {catboost_runtime:.2f} seconds")

CatBoost validation MAE: 1142.33
Runtime: 385.01 seconds


## XGBoostRegressor

In [18]:
X_train_xgb = X_train.copy()
X_valid_xgb = X_valid.copy()

for col in CAT_COLS:
    categories = pd.Index(X_train_xgb[col].astype(str).unique())

    X_train_xgb[col] = pd.Categorical(
        X_train_xgb[col].astype(str),
        categories=categories,
    )

    X_valid_xgb[col] = pd.Categorical(
        X_valid_xgb[col].astype(str),
        categories=categories,
    )

print(X_train_xgb.dtypes.value_counts())

category    72
float64     14
category    11
category     4
category     3
category     3
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
category     1
Name: count, dtype: int64


In [19]:
xgb_configs = [
    {
        "name": "xgb_1",
        "max_depth": 6,
        "learning_rate": 0.08,
        "n_estimators": 300,
    },
    {
        "name": "xgb_2",
        "max_depth": 8,
        "learning_rate": 0.05,
        "n_estimators": 400,
    },
]

In [20]:
xgb_cv_results = []

for config in xgb_configs:
    fold_maes = []
    fold_best_iterations = []
    start_time = time.time()

    print(f"\nRunning {config['name']}")

    for fold in range(5):
        fold_train_mask = cv_folds != fold
        fold_valid_mask = cv_folds == fold

        X_fold_train = X_train_xgb.loc[fold_train_mask]
        y_fold_train = y_train.loc[fold_train_mask]

        X_fold_valid = X_train_xgb.loc[fold_valid_mask]
        y_fold_valid = y_train.loc[fold_valid_mask]

        model = XGBRegressor(
            objective="reg:absoluteerror",
            eval_metric="mae",
            tree_method="hist",
            enable_categorical=True,
            max_depth=config["max_depth"],
            learning_rate=config["learning_rate"],
            n_estimators=config["n_estimators"],
            random_state=SEED,
            n_jobs=-1,
            early_stopping_rounds=30,
        )

        model.fit(
            X_fold_train,
            y_fold_train,
            eval_set=[(X_fold_valid, y_fold_valid)],
            verbose=False,
        )

        predictions = model.predict(X_fold_valid)

        fold_mae = mean_absolute_error(
            y_fold_valid,
            predictions,
        )

        fold_maes.append(fold_mae)
        fold_best_iterations.append(model.best_iteration)

        print(
            f"  Fold {fold}: "
            f"MAE={fold_mae:.2f}, "
            f"best_iter={model.best_iteration}"
        )

    runtime = time.time() - start_time

    xgb_cv_results.append(
        {
            **config,
            "cv_mae_mean": np.mean(fold_maes),
            "cv_mae_std": np.std(fold_maes),
            "mean_best_iteration": np.mean(fold_best_iterations),
            "runtime_seconds": runtime,
        }
    )

xgb_cv_df = pd.DataFrame(xgb_cv_results)

xgb_cv_df.sort_values("cv_mae_mean")


Running xgb_1
  Fold 0: MAE=1188.91, best_iter=299
  Fold 1: MAE=1181.25, best_iter=267
  Fold 2: MAE=1178.81, best_iter=299
  Fold 3: MAE=1193.24, best_iter=299
  Fold 4: MAE=1174.32, best_iter=213

Running xgb_2
  Fold 0: MAE=1189.91, best_iter=399
  Fold 1: MAE=1185.12, best_iter=399
  Fold 2: MAE=1183.91, best_iter=399
  Fold 3: MAE=1192.31, best_iter=399
  Fold 4: MAE=1173.96, best_iter=384


,name,max_depth,learning_rate,n_estimators,cv_mae_mean,cv_mae_std,mean_best_iteration,runtime_seconds
0,xgb_1,6,0.08,300,1183.307141,6.860628,275.4,141.665834
1,xgb_2,8,0.05,400,1185.041563,6.332161,396.0,187.813380


In [21]:
best_xgb_config = (
    xgb_cv_df
    .sort_values("cv_mae_mean")
    .iloc[0]
)

best_xgb_config

name                         xgb_1
max_depth                        6
learning_rate                 0.08
n_estimators                   300
cv_mae_mean            1183.307141
cv_mae_std                6.860628
mean_best_iteration          275.4
runtime_seconds         141.665834
Name: 0, dtype: object

In [22]:
xgb_start = time.time()

xgb_model = XGBRegressor(
    objective="reg:absoluteerror",
    eval_metric="mae",
    tree_method="hist",
    enable_categorical=True,
    max_depth=int(best_xgb_config["max_depth"]),
    learning_rate=float(best_xgb_config["learning_rate"]),
    n_estimators=int(best_xgb_config["n_estimators"]),
    random_state=SEED,
    n_jobs=-1,
)

xgb_model.fit(
    X_train_xgb,
    y_train,
    verbose=False,
)

xgb_runtime = time.time() - xgb_start

xgb_valid_pred = xgb_model.predict(X_valid_xgb)

xgb_valid_mae = mean_absolute_error(
    y_valid,
    xgb_valid_pred,
)

print(f"XGBoost validation MAE: {xgb_valid_mae:.2f}")
print(f"Runtime: {xgb_runtime:.2f} seconds")

XGBoost validation MAE: 1162.00
Runtime: 23.36 seconds


## Simple Version Comparison

In [23]:
full_feature_comparison = pd.DataFrame(
    [
        {
            "model": "CatBoostRegressor",
            "feature_count": 130,
            "cv_mae": best_catboost_config["cv_mae_mean"],
            "cv_std": best_catboost_config["cv_mae_std"],
            "validation_mae": catboost_valid_mae,
            "runtime_seconds": catboost_runtime,
        },
        {
            "model": "XGBRegressor",
            "feature_count": 130,
            "cv_mae": best_xgb_config["cv_mae_mean"],
            "cv_std": best_xgb_config["cv_mae_std"],
            "validation_mae": xgb_valid_mae,
            "runtime_seconds": xgb_runtime,
        },
    ]
)

full_feature_comparison["vs_mean_baseline_pct"] = (
    (MEAN_BASELINE_MAE - full_feature_comparison["validation_mae"])
    / MEAN_BASELINE_MAE
    * 100
)

full_feature_comparison["vs_median_baseline_pct"] = (
    (MEDIAN_BASELINE_MAE - full_feature_comparison["validation_mae"])
    / MEDIAN_BASELINE_MAE
    * 100
)

full_feature_comparison["vs_current_best"] = (
    full_feature_comparison["validation_mae"]
    - CURRENT_BEST_MAE
)

full_feature_comparison.sort_values("validation_mae")

,model,feature_count,cv_mae,cv_std,validation_mae,runtime_seconds,vs_mean_baseline_pct,vs_median_baseline_pct,vs_current_best
0,CatBoostRegressor,130,1159.997163,6.893022,1142.328900,385.007998,41.804573,36.716108,-8.841100
1,XGBRegressor,130,1183.307141,6.860628,1162.001325,23.361443,40.802370,35.626275,10.831325


In [24]:
if catboost_valid_mae <= xgb_valid_mae:
    importance_model_name = "CatBoostRegressor"
    importance_model = catboost_model
    importance_X = X_valid_cat
else:
    importance_model_name = "XGBRegressor"
    importance_model = xgb_model
    importance_X = X_valid_xgb

print("Importance model:", importance_model_name)

Importance model: CatBoostRegressor


In [25]:
if catboost_valid_mae <= xgb_valid_mae:
    importance_model_name = "CatBoostRegressor"
    importance_model = catboost_model
    importance_X = X_valid_cat
else:
    importance_model_name = "XGBRegressor"
    importance_model = xgb_model
    importance_X = X_valid_xgb

print("Importance model:", importance_model_name)

Importance model: CatBoostRegressor


In [34]:
if importance_model_name == "CatBoostRegressor":
    gain_importance = pd.Series(
        importance_model.get_feature_importance(),
        index=FEATURE_COLS,
        name="gain_importance",
    )

else:
    booster = importance_model.get_booster()

    gain_scores = booster.get_score(
        importance_type="gain"
    )

    gain_importance = pd.Series(
        {
            feature: gain_scores.get(feature, 0.0)
            for feature in FEATURE_COLS
        },
        name="gain_importance",
    )

gain_importance.sort_values(ascending=False).head(20)

cat80     12.811649
cat79      6.757488
cat57      5.684404
cat101     5.542758
cont7      5.300572
cat100     5.191694
cont2      4.952491
cat12      3.730406
cat114     3.589035
cat81      3.389281
cat111     3.067073
cat103     3.003567
cat113     2.818793
cat112     2.130879
cont14     1.936509
cont12     1.838383
cat53      1.744610
cat72      1.741709
cont11     1.683719
cat89      1.592623
Name: gain_importance, dtype: float64

In [26]:
PERMUTATION_SAMPLE_SIZE = 5000

perm_sample = validation.sample(
    n=min(PERMUTATION_SAMPLE_SIZE, len(validation)),
    random_state=SEED,
)

perm_y = perm_sample["loss"]

if importance_model_name == "CatBoostRegressor":
    perm_X = perm_sample[FEATURE_COLS].copy()

    for col in CAT_COLS:
        perm_X[col] = perm_X[col].astype(str)

else:
    perm_X = perm_sample[FEATURE_COLS].copy()

    for col in CAT_COLS:
        train_categories = X_train_xgb[col].cat.categories

        perm_X[col] = pd.Categorical(
            perm_X[col].astype(str),
            categories=train_categories,
        )

In [27]:
permutation_result = permutation_importance(
    importance_model,
    perm_X,
    perm_y,
    scoring="neg_mean_absolute_error",
    n_repeats=3,
    random_state=SEED,
    n_jobs=-1,
)

permutation_df = pd.DataFrame(
    {
        "feature": FEATURE_COLS,
        "permutation_importance": permutation_result.importances_mean,
        "permutation_std": permutation_result.importances_std,
    }
)

permutation_df.sort_values(
    "permutation_importance",
    ascending=False,
).head(20)

,feature,permutation_importance,permutation_std
100,cat101,124.559278,3.392511
79,cat80,104.225207,4.583087
78,cat79,55.436099,2.651625
102,cat103,48.772970,1.101303
113,cat114,48.450092,2.633088
56,cat57,40.988155,2.598894
99,cat100,38.963292,3.603050
117,cont2,37.345634,2.108851
80,cat81,35.054766,1.572314
110,cat111,34.782909,3.623519


In [ ]:
SHAP_SAMPLE_SIZE = 2000

shap_sample = validation.sample(
    n=min(SHAP_SAMPLE_SIZE, len(validation)),
    random_state=SEED,
)

if importance_model_name == "CatBoostRegressor":
    shap_X = shap_sample[FEATURE_COLS].copy()

    for col in CAT_COLS:
        shap_X[col] = shap_X[col].astype(str)

else:
    shap_X = shap_sample[FEATURE_COLS].copy()

    for col in CAT_COLS:
        shap_X[col] = pd.Categorical(
            shap_X[col].astype(str),
            categories=X_train_xgb[col].cat.categories,
        )

In [32]:
explainer = shap.TreeExplainer(importance_model)

shap_values = explainer.shap_values(shap_X)

shap_importance = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=FEATURE_COLS,
    name="shap_importance",
)

shap_importance.sort_values(
    ascending=False
).head(20)

cat80     371.557903
cat101    309.473943
cat79     230.787132
cat100    224.306752
cat81     201.214763
cat103    163.476359
cont2     156.120620
cat114    152.534200
cat111    122.643481
cat112    122.494557
cat72     121.046212
cat12     109.916394
cont7      89.937310
cat57      80.974336
cont14     72.203205
cat1       69.800834
cat87      68.518909
cat53      66.096518
cat108     52.050180
cat44      47.713790
Name: shap_importance, dtype: float64

In [35]:
feature_ranking = pd.DataFrame(
    {
        "feature": FEATURE_COLS,
        "model_importance": gain_importance.reindex(
            FEATURE_COLS
        ).fillna(0).values,
    }
)

feature_ranking = feature_ranking.merge(
    permutation_df[
        ["feature", "permutation_importance"]
    ],
    on="feature",
)

feature_ranking["shap_importance"] = (
    feature_ranking["feature"]
    .map(shap_importance)
    .fillna(0)
)

In [36]:
feature_ranking["model_rank"] = (
    feature_ranking["model_importance"]
    .rank(
        ascending=False,
        method="average",
    )
)

feature_ranking["permutation_rank"] = (
    feature_ranking["permutation_importance"]
    .rank(
        ascending=False,
        method="average",
    )
)

feature_ranking["shap_rank"] = (
    feature_ranking["shap_importance"]
    .rank(
        ascending=False,
        method="average",
    )
)

feature_ranking["aggregate_rank"] = (
    feature_ranking[
        [
            "model_rank",
            "permutation_rank",
            "shap_rank",
        ]
    ]
    .mean(axis=1)
)

feature_ranking = feature_ranking.sort_values(
    "aggregate_rank"
).reset_index(drop=True)

feature_ranking.head(30)

,feature,model_importance,permutation_importance,shap_importance,model_rank,permutation_rank,shap_rank,aggregate_rank
0,cat80,12.811649,104.225207,371.557903,1.0,2.0,1.0,1.333333
1,cat101,5.542758,124.559278,309.473943,4.0,1.0,2.0,2.333333
2,cat79,6.757488,55.436099,230.787132,2.0,3.0,3.0,2.666667
3,cat100,5.191694,38.963292,224.306752,6.0,7.0,4.0,5.666667
4,cont2,4.952491,37.345634,156.120620,7.0,8.0,7.0,7.333333
5,cat114,3.589035,48.450092,152.534200,9.0,5.0,8.0,7.333333
6,cat103,3.003567,48.772970,163.476359,12.0,4.0,6.0,7.333333
7,cat57,5.684404,40.988155,80.974336,3.0,6.0,14.0,7.666667
8,cat81,3.389281,35.054766,201.214763,10.0,9.0,5.0,8.000000
9,cont7,5.300572,21.423520,89.937310,5.0,12.0,13.0,10.000000


In [37]:
output_dir = (
    repo_root
    / "notebooks"
    / "liam-stapley"
    / "model_experiment_outputs"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True,
)

catboost_cv_df.to_csv(
    output_dir / "catboost_cv_results.csv",
    index=False,
)

xgb_cv_df.to_csv(
    output_dir / "xgboost_cv_results.csv",
    index=False,
)

full_feature_comparison.to_csv(
    output_dir / "full_feature_model_comparison.csv",
    index=False,
)

feature_ranking.to_csv(
    output_dir / "aggregate_feature_ranking.csv",
    index=False,
)

validation_predictions = pd.DataFrame(
    {
        "id": validation_ids.values,
        "loss": y_valid.values,
        "catboost_prediction": catboost_valid_pred,
        "xgboost_prediction": xgb_valid_pred,
    }
)

validation_predictions.to_csv(
    output_dir / "validation_predictions.csv",
    index=False,
)

print("Saved outputs to:", output_dir)

Saved outputs to: c:\Users\stapl\BTT\Fall AI Studio\Allstate 1A\Allstate-1A-predicting-auto-claims-severity\notebooks\liam-stapley\model_experiment_outputs


# Feature Selection Experiments

Using the aggregate feature ranking from the full-feature CatBoost model, this section tests smaller feature subsets while keeping the same frozen training folds and evaluation metric.

The goal is to determine whether removing lower-ranked features improves MAE, stability, or model simplicity.

Candidate subsets:
- Top 75 features
- Top 50 features
- Top 30 features

Subset selection is based only on the previously computed training-derived feature ranking. The three candidates are compared using the five frozen CV folds. Validation is used only after selecting the best reduced-feature configuration.

In [38]:
top_75_features = feature_ranking.head(75)["feature"].tolist()
top_50_features = feature_ranking.head(50)["feature"].tolist()
top_30_features = feature_ranking.head(30)["feature"].tolist()

feature_subsets = {
    "top_75": top_75_features,
    "top_50": top_50_features,
    "top_30": top_30_features,
}

for name, features in feature_subsets.items():
    print(name, len(features))

top_75 75
top_50 50
top_30 30


In [39]:
catboost_feature_selection_results = []

for subset_name, selected_features in feature_subsets.items():
    fold_maes = []
    fold_best_iterations = []
    start_time = time.time()

    selected_cat_cols = [
        col for col in selected_features
        if col in CAT_COLS
    ]

    cat_indices = [
        selected_features.index(col)
        for col in selected_cat_cols
    ]

    print(f"\nRunning {subset_name}")

    for fold in range(5):
        fold_train_mask = cv_folds != fold
        fold_valid_mask = cv_folds == fold

        X_fold_train = X_train_cat.loc[
            fold_train_mask,
            selected_features,
        ]

        y_fold_train = y_train.loc[
            fold_train_mask
        ]

        X_fold_valid = X_train_cat.loc[
            fold_valid_mask,
            selected_features,
        ]

        y_fold_valid = y_train.loc[
            fold_valid_mask
        ]

        model = CatBoostRegressor(
            loss_function="MAE",
            eval_metric="MAE",
            depth=int(best_catboost_config["depth"]),
            learning_rate=float(
                best_catboost_config["learning_rate"]
            ),
            iterations=int(
                best_catboost_config["iterations"]
            ),
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )

        model.fit(
            X_fold_train,
            y_fold_train,
            cat_features=cat_indices,
            eval_set=(
                X_fold_valid,
                y_fold_valid,
            ),
            early_stopping_rounds=30,
            use_best_model=True,
        )

        predictions = model.predict(
            X_fold_valid
        )

        fold_mae = mean_absolute_error(
            y_fold_valid,
            predictions,
        )

        fold_maes.append(fold_mae)
        fold_best_iterations.append(
            model.get_best_iteration()
        )

        print(
            f"  Fold {fold}: "
            f"MAE={fold_mae:.2f}, "
            f"best_iter={model.get_best_iteration()}"
        )

    runtime = time.time() - start_time

    catboost_feature_selection_results.append(
        {
            "subset": subset_name,
            "feature_count": len(selected_features),
            "cv_mae_mean": np.mean(fold_maes),
            "cv_mae_std": np.std(fold_maes),
            "mean_best_iteration": np.mean(
                fold_best_iterations
            ),
            "runtime_seconds": runtime,
        }
    )

catboost_feature_selection_df = pd.DataFrame(
    catboost_feature_selection_results
)

catboost_feature_selection_df.sort_values(
    "cv_mae_mean"
)


Running top_75
  Fold 0: MAE=1170.68, best_iter=249
  Fold 1: MAE=1157.16, best_iter=249
  Fold 2: MAE=1154.79, best_iter=249
  Fold 3: MAE=1166.30, best_iter=249
  Fold 4: MAE=1150.02, best_iter=249

Running top_50
  Fold 0: MAE=1169.63, best_iter=249
  Fold 1: MAE=1158.18, best_iter=249
  Fold 2: MAE=1155.98, best_iter=249
  Fold 3: MAE=1168.54, best_iter=249
  Fold 4: MAE=1153.28, best_iter=249

Running top_30
  Fold 0: MAE=1175.60, best_iter=249
  Fold 1: MAE=1167.98, best_iter=248
  Fold 2: MAE=1160.55, best_iter=249
  Fold 3: MAE=1173.43, best_iter=247
  Fold 4: MAE=1158.19, best_iter=246


,subset,feature_count,cv_mae_mean,cv_mae_std,mean_best_iteration,runtime_seconds
0,top_75,75,1159.790004,7.594679,249.0,1714.236942
1,top_50,50,1161.122857,6.692453,249.0,1147.795882
2,top_30,30,1167.149249,6.862526,247.8,737.552782


In [40]:
full_catboost_row = {
    "subset": "all_130",
    "feature_count": 130,
    "cv_mae_mean": float(
        best_catboost_config["cv_mae_mean"]
    ),
    "cv_mae_std": float(
        best_catboost_config["cv_mae_std"]
    ),
    "mean_best_iteration": float(
        best_catboost_config[
            "mean_best_iteration"
        ]
    ),
    "runtime_seconds": float(
        best_catboost_config[
            "runtime_seconds"
        ]
    ),
}

feature_selection_comparison = pd.concat(
    [
        pd.DataFrame([full_catboost_row]),
        catboost_feature_selection_df,
    ],
    ignore_index=True,
)

feature_selection_comparison.sort_values(
    "cv_mae_mean"
)

,subset,feature_count,cv_mae_mean,cv_mae_std,mean_best_iteration,runtime_seconds
1,top_75,75,1159.790004,7.594679,249.0,1714.236942
0,all_130,130,1159.997163,6.893022,249.0,1393.581232
2,top_50,50,1161.122857,6.692453,249.0,1147.795882
3,top_30,30,1167.149249,6.862526,247.8,737.552782


In [41]:
best_reduced = (
    catboost_feature_selection_df
    .sort_values("cv_mae_mean")
    .iloc[0]
)

best_reduced

subset                      top_75
feature_count                   75
cv_mae_mean            1159.790004
cv_mae_std                7.594679
mean_best_iteration          249.0
runtime_seconds        1714.236942
Name: 0, dtype: object

In [42]:
best_subset_name = best_reduced["subset"]

best_subset_features = feature_subsets[
    best_subset_name
]

best_subset_cat_cols = [
    col for col in best_subset_features
    if col in CAT_COLS
]

best_subset_cat_indices = [
    best_subset_features.index(col)
    for col in best_subset_cat_cols
]

reduced_catboost_model = CatBoostRegressor(
    loss_function="MAE",
    eval_metric="MAE",
    depth=int(best_catboost_config["depth"]),
    learning_rate=float(
        best_catboost_config["learning_rate"]
    ),
    iterations=int(
        best_catboost_config["iterations"]
    ),
    random_seed=SEED,
    verbose=False,
    allow_writing_files=False,
    thread_count=-1,
)

reduced_catboost_model.fit(
    X_train_cat[best_subset_features],
    y_train,
    cat_features=best_subset_cat_indices,
)

reduced_valid_pred = reduced_catboost_model.predict(
    X_valid_cat[best_subset_features]
)

reduced_valid_mae = mean_absolute_error(
    y_valid,
    reduced_valid_pred,
)

print("Selected subset:", best_subset_name)
print("Feature count:", len(best_subset_features))
print(
    f"Reduced CatBoost validation MAE: "
    f"{reduced_valid_mae:.2f}"
)
print(
    f"Full CatBoost validation MAE: "
    f"{catboost_valid_mae:.2f}"
)

Selected subset: top_75
Feature count: 75
Reduced CatBoost validation MAE: 1141.14
Full CatBoost validation MAE: 1142.33


In [43]:
feature_selection_comparison.to_csv(
    output_dir / "catboost_feature_selection_cv.csv",
    index=False,
)

pd.DataFrame(
    {
        "id": validation_ids.values,
        "loss": y_valid.values,
        "full_catboost_prediction": catboost_valid_pred,
        "reduced_catboost_prediction": reduced_valid_pred,
    }
).to_csv(
    output_dir / "catboost_feature_selection_predictions.csv",
    index=False,
)

# Final Robustness Checks

The current best model is CatBoost using the top 75 ranked features.

Before selecting the final model, two small training-only checks are performed:

1. A limited CatBoost tuning check.
2. A robust-loss experiment using Huber loss to reduce the influence of extreme residuals.

To limit computation and avoid excessive tuning, candidate configurations are first screened on two fixed CV folds. Only a candidate that materially improves on the current CatBoost CV performance is promoted to full five-fold evaluation.

In [44]:
SCREENING_FOLDS = [0, 1]

CURRENT_BEST_CV_MAE = float(
    feature_selection_comparison
    .loc[
        feature_selection_comparison["subset"] == "top_75",
        "cv_mae_mean"
    ]
    .iloc[0]
)

PROMOTION_THRESHOLD = 0.005  # require at least 0.5% improvement

print(f"Current best CV MAE: {CURRENT_BEST_CV_MAE:.2f}")
print(f"Promotion threshold: {PROMOTION_THRESHOLD * 100:.1f}%")

Current best CV MAE: 1159.79
Promotion threshold: 0.5%


In [45]:
final_check_configs = [
    {
        "name": "current_top75_mae",
        "loss_function": "MAE",
        "depth": int(best_catboost_config["depth"]),
        "learning_rate": float(best_catboost_config["learning_rate"]),
        "iterations": 250,
    },
    {
        "name": "top75_mae_more_iterations",
        "loss_function": "MAE",
        "depth": int(best_catboost_config["depth"]),
        "learning_rate": float(best_catboost_config["learning_rate"]),
        "iterations": 500,
    },
    {
        "name": "top75_huber",
        "loss_function": "Huber:delta=1000",
        "depth": int(best_catboost_config["depth"]),
        "learning_rate": float(best_catboost_config["learning_rate"]),
        "iterations": 500,
    },
]

In [46]:
final_screen_results = []

selected_features = top_75_features

selected_cat_cols = [
    col for col in selected_features
    if col in CAT_COLS
]

selected_cat_indices = [
    selected_features.index(col)
    for col in selected_cat_cols
]

for config in final_check_configs:
    fold_maes = []
    fold_best_iterations = []
    start_time = time.time()

    print(f"\nScreening {config['name']}")

    for fold in SCREENING_FOLDS:
        fold_train_mask = cv_folds != fold
        fold_valid_mask = cv_folds == fold

        X_fold_train = X_train_cat.loc[
            fold_train_mask,
            selected_features,
        ]

        y_fold_train = y_train.loc[
            fold_train_mask
        ]

        X_fold_valid = X_train_cat.loc[
            fold_valid_mask,
            selected_features,
        ]

        y_fold_valid = y_train.loc[
            fold_valid_mask
        ]

        model = CatBoostRegressor(
            loss_function=config["loss_function"],
            eval_metric="MAE",
            depth=config["depth"],
            learning_rate=config["learning_rate"],
            iterations=config["iterations"],
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )

        model.fit(
            X_fold_train,
            y_fold_train,
            cat_features=selected_cat_indices,
            eval_set=(X_fold_valid, y_fold_valid),
            early_stopping_rounds=30,
            use_best_model=True,
        )

        predictions = model.predict(X_fold_valid)

        fold_mae = mean_absolute_error(
            y_fold_valid,
            predictions,
        )

        fold_maes.append(fold_mae)
        fold_best_iterations.append(
            model.get_best_iteration()
        )

        print(
            f"  Fold {fold}: "
            f"MAE={fold_mae:.2f}, "
            f"best_iter={model.get_best_iteration()}"
        )

    runtime = time.time() - start_time

    mean_mae = np.mean(fold_maes)

    final_screen_results.append(
        {
            **config,
            "screen_cv_mae": mean_mae,
            "screen_cv_std": np.std(fold_maes),
            "mean_best_iteration": np.mean(
                fold_best_iterations
            ),
            "runtime_seconds": runtime,
        }
    )

final_screen_df = pd.DataFrame(final_screen_results)

final_screen_df.sort_values("screen_cv_mae")


Screening current_top75_mae
  Fold 0: MAE=1170.68, best_iter=249
  Fold 1: MAE=1157.16, best_iter=249

Screening top75_mae_more_iterations
  Fold 0: MAE=1163.14, best_iter=497
  Fold 1: MAE=1151.37, best_iter=494

Screening top75_huber
  Fold 0: MAE=1165.75, best_iter=499
  Fold 1: MAE=1154.83, best_iter=499


,name,loss_function,depth,learning_rate,iterations,screen_cv_mae,screen_cv_std,mean_best_iteration,runtime_seconds
1,top75_mae_more_iterations,MAE,8,0.08,500,1157.251766,5.884080,495.5,2196.221726
2,top75_huber,Huber:delta=1000,8,0.08,500,1160.288503,5.460315,499.0,1537.334242
0,current_top75_mae,MAE,8,0.08,250,1163.923389,6.760586,249.0,847.241557


In [47]:
screen_baseline = (
    final_screen_df
    .loc[
        final_screen_df["name"] == "current_top75_mae",
        "screen_cv_mae"
    ]
    .iloc[0]
)

final_screen_df["improvement_pct"] = (
    (screen_baseline - final_screen_df["screen_cv_mae"])
    / screen_baseline
    * 100
)

promotion_candidates = final_screen_df[
    (
        final_screen_df["name"] != "current_top75_mae"
    )
    & (
        final_screen_df["improvement_pct"]
        >= PROMOTION_THRESHOLD * 100
    )
].sort_values("screen_cv_mae")

final_screen_df[
    [
        "name",
        "screen_cv_mae",
        "screen_cv_std",
        "improvement_pct",
        "mean_best_iteration",
    ]
]

,name,screen_cv_mae,screen_cv_std,improvement_pct,mean_best_iteration
0,current_top75_mae,1163.923389,6.760586,0.000000,249.0
1,top75_mae_more_iterations,1157.251766,5.884080,0.573201,495.5
2,top75_huber,1160.288503,5.460315,0.312296,499.0


In [48]:
if promotion_candidates.empty:
    print(
        "No candidate improved enough during screening. "
        "Keep the existing top-75 CatBoost model."
    )
    promoted_config = None
else:
    promoted_config = promotion_candidates.iloc[0].to_dict()

    print("Promoted candidate:")
    print(promoted_config["name"])

Promoted candidate:
top75_mae_more_iterations


In [49]:
promoted_cv_result = None

if promoted_config is not None:
    fold_maes = []
    fold_best_iterations = []
    start_time = time.time()

    print(
        f"\nRunning full 5-fold CV for "
        f"{promoted_config['name']}"
    )

    for fold in range(5):
        fold_train_mask = cv_folds != fold
        fold_valid_mask = cv_folds == fold

        X_fold_train = X_train_cat.loc[
            fold_train_mask,
            selected_features,
        ]
        y_fold_train = y_train.loc[
            fold_train_mask
        ]

        X_fold_valid = X_train_cat.loc[
            fold_valid_mask,
            selected_features,
        ]
        y_fold_valid = y_train.loc[
            fold_valid_mask
        ]

        model = CatBoostRegressor(
            loss_function=promoted_config[
                "loss_function"
            ],
            eval_metric="MAE",
            depth=int(promoted_config["depth"]),
            learning_rate=float(
                promoted_config["learning_rate"]
            ),
            iterations=int(
                promoted_config["iterations"]
            ),
            random_seed=SEED,
            verbose=False,
            allow_writing_files=False,
            thread_count=-1,
        )

        model.fit(
            X_fold_train,
            y_fold_train,
            cat_features=selected_cat_indices,
            eval_set=(X_fold_valid, y_fold_valid),
            early_stopping_rounds=30,
            use_best_model=True,
        )

        predictions = model.predict(X_fold_valid)

        fold_mae = mean_absolute_error(
            y_fold_valid,
            predictions,
        )

        fold_maes.append(fold_mae)
        fold_best_iterations.append(
            model.get_best_iteration()
        )

        print(
            f"  Fold {fold}: "
            f"MAE={fold_mae:.2f}, "
            f"best_iter={model.get_best_iteration()}"
        )

    promoted_cv_result = {
        "name": promoted_config["name"],
        "cv_mae_mean": np.mean(fold_maes),
        "cv_mae_std": np.std(fold_maes),
        "mean_best_iteration": np.mean(
            fold_best_iterations
        ),
        "runtime_seconds": time.time() - start_time,
    }

    print("\nFull CV result:")
    print(promoted_cv_result)


Running full 5-fold CV for top75_mae_more_iterations
  Fold 0: MAE=1163.14, best_iter=497
  Fold 1: MAE=1151.37, best_iter=494
  Fold 2: MAE=1148.72, best_iter=499
  Fold 3: MAE=1160.58, best_iter=499
  Fold 4: MAE=1144.11, best_iter=497

Full CV result:
{'name': 'top75_mae_more_iterations', 'cv_mae_mean': 1153.5829266911792, 'cv_mae_std': 7.189688568401656, 'mean_best_iteration': 497.2, 'runtime_seconds': 7333.775911092758}


In [50]:
if promoted_cv_result is None:
    final_model_choice = "top75_catboost_mae"

    print(
        "FINAL CHOICE: top-75 CatBoost with MAE loss"
    )

elif (
    promoted_cv_result["cv_mae_mean"]
    < CURRENT_BEST_CV_MAE
):
    final_model_choice = promoted_cv_result["name"]

    print(
        f"FINAL CHOICE: {final_model_choice}"
    )

else:
    final_model_choice = "top75_catboost_mae"

    print(
        "Promoted model did not beat the current "
        "top-75 CatBoost across all five folds."
    )
    print(
        "FINAL CHOICE: top-75 CatBoost with MAE loss"
    )

FINAL CHOICE: top75_mae_more_iterations


In [51]:
final_cv_results = pd.DataFrame([
    {
        "model": "CatBoostRegressor",
        "variant": "top75_mae_more_iterations",
        "feature_count": 75,
        "cv_mae_mean": promoted_cv_result["cv_mae_mean"],
        "cv_mae_std": promoted_cv_result["cv_mae_std"],
        "mean_best_iteration": promoted_cv_result["mean_best_iteration"],
        "runtime_seconds": promoted_cv_result["runtime_seconds"],
    }
])

final_cv_results.to_csv(
    output_dir / "final_catboost_cv_results.csv",
    index=False,
)

final_cv_results

,model,variant,feature_count,cv_mae_mean,cv_mae_std,mean_best_iteration,runtime_seconds
0,CatBoostRegressor,top75_mae_more_iterations,75,1153.582927,7.189689,497.2,7333.775911


In [52]:
final_catboost_model = CatBoostRegressor(
    loss_function="MAE",
    eval_metric="MAE",
    depth=int(promoted_config["depth"]),
    learning_rate=float(promoted_config["learning_rate"]),
    iterations=500,
    random_seed=SEED,
    verbose=False,
    allow_writing_files=False,
    thread_count=-1,
)

final_catboost_model.fit(
    X_train_cat[top_75_features],
    y_train,
    cat_features=selected_cat_indices,
)

CatBoostRegressor(allow_writing_files=False, depth=8, eval_metric='MAE', iterations=500, learning_rate=0.08, loss_function='MAE', random_seed=42, verbose=False)

In [53]:
final_model_path = (
    output_dir / "final_top75_catboost_model.cbm"
)

final_catboost_model.save_model(
    str(final_model_path)
)

print("Saved model to:", final_model_path)

Saved model to: c:\Users\stapl\BTT\Fall AI Studio\Allstate 1A\Allstate-1A-predicting-auto-claims-severity\notebooks\liam-stapley\model_experiment_outputs\final_top75_catboost_model.cbm


In [54]:
loaded_catboost_model = CatBoostRegressor()

loaded_catboost_model.load_model(
    str(final_model_path)
)

print("Model reload successful.")

Model reload successful.


In [55]:
final_valid_pred = final_catboost_model.predict(
    X_valid_cat[top_75_features]
)

final_valid_mae = mean_absolute_error(
    y_valid,
    final_valid_pred,
)

print(
    f"Final CatBoost validation MAE: "
    f"{final_valid_mae:.2f}"
)

Final CatBoost validation MAE: 1136.34


In [56]:
final_validation_predictions = pd.DataFrame(
    {
        "id": validation_ids.values,
        "loss": y_valid.values,
        "prediction": final_valid_pred,
        "absolute_error": np.abs(
            y_valid.values - final_valid_pred
        ),
    }
)

final_validation_predictions.to_csv(
    output_dir / "final_catboost_validation_predictions.csv",
    index=False,
)

In [57]:
pd.Series(
    top_75_features,
    name="feature"
).to_csv(
    output_dir / "final_catboost_features.csv",
    index=False,
)

In [58]:
train_p90 = y_train.quantile(0.90)
train_p99 = y_train.quantile(0.99)

final_eval = pd.DataFrame(
    {
        "id": validation_ids.values,
        "loss": y_valid.values,
        "prediction": final_valid_pred,
    }
)

final_eval["absolute_error"] = np.abs(
    final_eval["loss"] - final_eval["prediction"]
)

segments = [
    (
        "loss < p90",
        final_eval["loss"] < train_p90,
    ),
    (
        "p90 to p99",
        (final_eval["loss"] >= train_p90)
        & (final_eval["loss"] < train_p99),
    ),
    (
        "loss >= p99",
        final_eval["loss"] >= train_p99,
    ),
]

tail_rows = []

for segment_name, mask in segments:
    segment = final_eval.loc[mask]

    tail_rows.append(
        {
            "segment": segment_name,
            "support": len(segment),
            "mae": segment["absolute_error"].mean(),
        }
    )

final_tail_diagnostics = pd.DataFrame(tail_rows)

final_tail_diagnostics

,segment,support,mae
0,loss < p90,25422,868.051743
1,p90 to p99,2547,2976.906678
2,loss >= p99,279,8779.901930


In [59]:
final_model_summary = pd.DataFrame(
    [
        {
            "model": "CatBoostRegressor",
            "variant": "top75_mae_500_iterations",
            "feature_count": 75,
            "cv_mae_mean": promoted_cv_result["cv_mae_mean"],
            "cv_mae_std": promoted_cv_result["cv_mae_std"],
            "mean_best_iteration": promoted_cv_result["mean_best_iteration"],
            "validation_mae": final_valid_mae,
            "validation_claims": len(y_valid),
            "vs_mean_baseline_pct": (
                (MEAN_BASELINE_MAE - final_valid_mae)
                / MEAN_BASELINE_MAE
                * 100
            ),
            "vs_median_baseline_pct": (
                (MEDIAN_BASELINE_MAE - final_valid_mae)
                / MEDIAN_BASELINE_MAE
                * 100
            ),
            "vs_previous_best_hgb": (
                CURRENT_BEST_MAE - final_valid_mae
            ),
        }
    ]
)

final_model_summary.to_csv(
    output_dir / "final_catboost_summary.csv",
    index=False,
)

final_tail_diagnostics.to_csv(
    output_dir / "final_catboost_tail_diagnostics.csv",
    index=False,
)

final_eval.to_csv(
    output_dir / "final_catboost_validation_predictions.csv",
    index=False,
)

pd.Series(
    top_75_features,
    name="feature",
).to_csv(
    output_dir / "final_catboost_features.csv",
    index=False,
)

final_model_summary

,model,variant,feature_count,cv_mae_mean,cv_mae_std,mean_best_iteration,validation_mae,validation_claims,vs_mean_baseline_pct,vs_median_baseline_pct,vs_previous_best_hgb
0,CatBoostRegressor,top75_mae_500_iterations,75,1153.582927,7.189689,497.2,1136.341878,28248,42.109579,37.047783,14.828122
